# 07.1 Benchmarking LLM Inference

Async benchmark client measuring TTFT, TBT, E2E latency.
Workload generation, SLO validation, capacity planning, bottleneck diagnostics.


In [ ]:
import sys
sys.path.insert(0, '../../..')

import asyncio
import time
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from content.utils.latency import LatencyTracker
from content.utils.benchmark import BenchmarkHarness


In [ ]:
@dataclass
class RequestMetrics:
    ttft: float = 0.0  # Time to first token (s)
    tbt: List[float] = field(default_factory=list)  # Inter-token latencies
    e2e: float = 0.0  # End-to-end latency (s)
    tokens_generated: int = 0
    input_tokens: int = 0
    success: bool = True
    error: Optional[str] = None

    @property
    def mean_tbt(self) -> float:
        return np.mean(self.tbt) if self.tbt else 0.0

    @property
    def p99_tbt(self) -> float:
        return np.percentile(self.tbt, 99) if self.tbt else 0.0


In [ ]:
class AsyncBenchmarkClient:
    """Async client that measures TTFT/TBT/E2E against streaming endpoints."""

    def __init__(self, base_url: str = "http://localhost:8000", model: str = "default"):
        self.base_url = base_url
        self.model = model
        self.results: List[RequestMetrics] = []

    async def send_request(self, prompt: str, max_tokens: int = 128) -> RequestMetrics:
        """Send a single streaming request and measure latencies."""
        metrics = RequestMetrics(input_tokens=len(prompt.split()))
        try:
            import aiohttp
            payload = {"model": self.model, "prompt": prompt,
                       "max_tokens": max_tokens, "stream": True}
            start = time.perf_counter()
            first_token = False
            last_token_time = start

            async with aiohttp.ClientSession() as session:
                async with session.post(f"{self.base_url}/v1/completions", json=payload) as resp:
                    async for chunk in resp.content:
                        now = time.perf_counter()
                        if not first_token:
                            metrics.ttft = now - start
                            first_token = True
                        else:
                            metrics.tbt.append(now - last_token_time)
                        last_token_time = now
                        metrics.tokens_generated += 1

            metrics.e2e = time.perf_counter() - start
        except Exception as e:
            metrics.success = False
            metrics.error = str(e)
        self.results.append(metrics)
        return metrics

    async def run_batch(self, prompts: List[str], max_tokens: int = 128,
                        concurrency: int = 8) -> List[RequestMetrics]:
        """Run batch of requests with bounded concurrency."""
        sem = asyncio.Semaphore(concurrency)
        async def bounded(p):
            async with sem:
                return await self.send_request(p, max_tokens)
        return await asyncio.gather(*[bounded(p) for p in prompts])


In [ ]:
class WorkloadGenerator:
    """Generate realistic workloads with configurable distributions."""

    PROFILES = {
        "chatbot": {"input_range": (20, 200), "output_tokens": 128, "think_time_ms": 500},
        "code": {"input_range": (100, 2000), "output_tokens": 512, "think_time_ms": 2000},
        "batch": {"input_range": (500, 4000), "output_tokens": 256, "think_time_ms": 0},
        "voice": {"input_range": (5, 50), "output_tokens": 64, "think_time_ms": 100},
    }

    def __init__(self, profile: str = "chatbot", rng_seed: int = 42):
        self.profile = self.PROFILES[profile]
        self.rng = np.random.default_rng(rng_seed)

    def generate(self, n: int) -> List[str]:
        """Generate n prompts with token counts drawn from profile distribution."""
        lo, hi = self.profile["input_range"]
        lengths = self.rng.integers(lo, hi, size=n)
        return [" ".join(["token"] * l) for l in lengths]

    def arrival_times(self, n: int, rps: float) -> List[float]:
        """Poisson arrival times for n requests at given RPS."""
        intervals = self.rng.exponential(1.0 / rps, size=n)
        return np.cumsum(intervals).tolist()


# Demo
wg = WorkloadGenerator("chatbot")
prompts = wg.generate(5)
print(f"Generated {len(prompts)} prompts, lengths: {[len(p.split()) for p in prompts]}")
print(f"Arrival times at 10 RPS: {wg.arrival_times(5, 10.0)}")


## Latency Distribution Analysis

Visualize TTFT, TBT, and E2E distributions with percentile markers.


In [ ]:
# Simulate benchmark results for demonstration (replace with real client.run_batch)
rng = np.random.default_rng(42)
n_requests = 500

sim_results = []
for _ in range(n_requests):
    ttft = rng.lognormal(np.log(0.08), 0.4)
    n_tok = rng.integers(30, 200)
    tbt = rng.lognormal(np.log(0.025), 0.3, size=n_tok).tolist()
    e2e = ttft + sum(tbt)
    sim_results.append(RequestMetrics(ttft=ttft, tbt=tbt, e2e=e2e,
                                      tokens_generated=n_tok, input_tokens=rng.integers(20, 500)))

print(f"Simulated {len(sim_results)} requests")


In [ ]:
def plot_latency_distributions(results: List[RequestMetrics]):
    """Plot TTFT, mean TBT, and E2E latency distributions."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    ttfts = [r.ttft * 1000 for r in results if r.success]
    mean_tbts = [r.mean_tbt * 1000 for r in results if r.success and r.tbt]
    e2es = [r.e2e * 1000 for r in results if r.success]

    for ax, data, title in zip(axes, [ttfts, mean_tbts, e2es],
                                ["TTFT (ms)", "Mean TBT (ms)", "E2E (ms)"]):
        ax.hist(data, bins=50, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
        for pct, color in [(50, 'green'), (95, 'orange'), (99, 'red')]:
            val = np.percentile(data, pct)
            ax.axvline(val, color=color, linestyle='--', label=f'P{pct}: {val:.1f}ms')
        ax.set_title(title)
        ax.set_xlabel('Latency (ms)')
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

plot_latency_distributions(sim_results)


## SLO Validation

Define per-use-case SLOs and validate benchmark results against them.


In [ ]:
@dataclass
class SLO:
    name: str
    ttft_p99_ms: float
    tbt_p99_ms: float
    e2e_p99_ms: float
    error_rate_pct: float = 1.0


SLO_PROFILES = {
    "chatbot": SLO("chatbot", ttft_p99_ms=300, tbt_p99_ms=80, e2e_p99_ms=5000, error_rate_pct=0.5),
    "code": SLO("code", ttft_p99_ms=1000, tbt_p99_ms=100, e2e_p99_ms=30000, error_rate_pct=1.0),
    "batch": SLO("batch", ttft_p99_ms=5000, tbt_p99_ms=200, e2e_p99_ms=120000, error_rate_pct=2.0),
    "voice": SLO("voice", ttft_p99_ms=150, tbt_p99_ms=50, e2e_p99_ms=2000, error_rate_pct=0.1),
}


def validate_slo(results: List[RequestMetrics], slo: SLO) -> Dict[str, any]:
    """Validate results against an SLO. Returns pass/fail per metric."""
    successful = [r for r in results if r.success]
    error_rate = (1 - len(successful) / len(results)) * 100 if results else 100

    ttft_p99 = np.percentile([r.ttft * 1000 for r in successful], 99)
    tbt_p99 = np.percentile([r.p99_tbt * 1000 for r in successful if r.tbt], 99)
    e2e_p99 = np.percentile([r.e2e * 1000 for r in successful], 99)

    return {
        "slo": slo.name,
        "ttft_p99": {"actual": ttft_p99, "limit": slo.ttft_p99_ms, "pass": ttft_p99 <= slo.ttft_p99_ms},
        "tbt_p99": {"actual": tbt_p99, "limit": slo.tbt_p99_ms, "pass": tbt_p99 <= slo.tbt_p99_ms},
        "e2e_p99": {"actual": e2e_p99, "limit": slo.e2e_p99_ms, "pass": e2e_p99 <= slo.e2e_p99_ms},
        "error_rate": {"actual": error_rate, "limit": slo.error_rate_pct, "pass": error_rate <= slo.error_rate_pct},
    }


In [ ]:
# Validate against all SLO profiles
for profile_name, slo in SLO_PROFILES.items():
    result = validate_slo(sim_results, slo)
    status = '✅' if all(v['pass'] for v in result.values() if isinstance(v, dict)) else '❌'
    print(f"{status} {profile_name.upper()} SLO:")
    for metric, v in result.items():
        if isinstance(v, dict):
            flag = '✓' if v['pass'] else '✗'
            print(f"   {flag} {metric}: {v['actual']:.1f} / {v['limit']:.1f} ms")
    print()


## Capacity Planning

Estimate required GPU count given target throughput and measured per-GPU capacity.


In [ ]:
def capacity_plan(target_rps: float, results: List[RequestMetrics],
                  gpu_count_tested: int = 1, headroom: float = 0.3) -> Dict:
    """Estimate GPUs needed for target RPS with headroom."""
    successful = [r for r in results if r.success]
    avg_e2e = np.mean([r.e2e for r in successful])
    # Max concurrent requests per GPU ≈ test_concurrency that kept SLO
    measured_rps = len(successful) / (max(r.e2e for r in successful))  # approx
    rps_per_gpu = measured_rps / gpu_count_tested

    gpus_raw = target_rps / rps_per_gpu
    gpus_with_headroom = gpus_raw * (1 + headroom)

    return {
        "measured_rps_per_gpu": rps_per_gpu,
        "target_rps": target_rps,
        "gpus_needed_raw": int(np.ceil(gpus_raw)),
        "gpus_with_headroom": int(np.ceil(gpus_with_headroom)),
        "avg_e2e_s": avg_e2e,
        "headroom_pct": headroom * 100,
    }


plan = capacity_plan(target_rps=100, results=sim_results, gpu_count_tested=1)
for k, v in plan.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")


## Bottleneck Diagnostic

Identify whether the system is prefill-bound, decode-bound, or scheduling-bound.


In [ ]:
def diagnose_bottleneck(results: List[RequestMetrics]) -> Dict:
    """Classify bottleneck from latency breakdown."""
    successful = [r for r in results if r.success and r.tbt]
    ttfts = np.array([r.ttft for r in successful])
    decode_times = np.array([sum(r.tbt) for r in successful])
    e2es = np.array([r.e2e for r in successful])

    prefill_frac = np.mean(ttfts / e2es)
    decode_frac = np.mean(decode_times / e2es)
    overhead_frac = 1 - prefill_frac - decode_frac

    # Correlation: if TTFT variance is high relative to decode, prefill is bottleneck
    ttft_cv = np.std(ttfts) / np.mean(ttfts)
    decode_cv = np.std(decode_times) / np.mean(decode_times)

    if prefill_frac > 0.4 or ttft_cv > decode_cv * 1.5:
        bottleneck = "PREFILL-BOUND (compute-limited on prompt processing)"
    elif overhead_frac > 0.2:
        bottleneck = "SCHEDULING-BOUND (queuing/batching overhead)"
    else:
        bottleneck = "DECODE-BOUND (memory-bandwidth limited on token generation)"

    return {
        "bottleneck": bottleneck,
        "prefill_fraction": f"{prefill_frac:.1%}",
        "decode_fraction": f"{decode_frac:.1%}",
        "overhead_fraction": f"{overhead_frac:.1%}",
        "ttft_cv": f"{ttft_cv:.3f}",
        "decode_cv": f"{decode_cv:.3f}",
    }


diag = diagnose_bottleneck(sim_results)
print(f"Diagnosis: {diag['bottleneck']}")
print(f"  Prefill: {diag['prefill_fraction']} | Decode: {diag['decode_fraction']} | Overhead: {diag['overhead_fraction']}")
print(f"  TTFT CV: {diag['ttft_cv']} | Decode CV: {diag['decode_cv']}")


## Key Takeaways

1. **TTFT** dominated by prefill compute — scales with input length
2. **TBT** bounded by memory bandwidth — use continuous batching to amortize
3. **SLO profiles** differ 10x+ between voice (150ms TTFT) and batch (5s TTFT)
4. **Capacity planning** must include 30%+ headroom for traffic spikes
5. **Bottleneck diagnosis** guides optimization: prefill-bound → chunked prefill, decode-bound → speculative decoding
